# Three-arm control: all test regimes

Run the oracle, matched-size historical and full-history comparison for
2018, 2020, 2022 and 2024. The wrapper reads the experiment functions from
the source notebooks and writes `new-reward/checkpoints/e2_summary.csv`
for Table 2. Run it after the Layer-1 panel notebook.

Start Jupyter from the repository root.


In [ ]:
# ══ E2 AUTORUN: all four regimes, three arms, one click ══
RUN_REGIMES = ['mix_test2022', 'crash_test2018', 'crash_test2020', 'mix_test2024']
SPLIT_MODE = 'interleave2'
E2_SEEDS = [0, 1, 2, 3, 4]
SKIP_DONE = True

from pathlib import Path
import os
import sys
import runpy


def _find_repository_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'ctc_runtime.py').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the conformal-trust-control repository.')

KIT_ROOT = _find_repository_root()

_KIT = KIT_ROOT
NB_V12 = _KIT / 'notebooks' / '06_common_budget_setup.ipynb'
NB_PORTABLE_SETUP = _KIT / 'notebooks' / '03_evaluate_policy_arms.ipynb'
assert NB_V12.is_file(), f'sealed v1.2 source not found: {NB_V12}'
assert NB_PORTABLE_SETUP.is_file(), f'portable setup source not found: {NB_PORTABLE_SETUP}'
NB_E2 = KIT_ROOT / 'notebooks' / '04_three_arm_setup.ipynb'
assert NB_E2.is_file(), f'E2 source not found: {NB_E2}'

import glob as _g, json as _j, os as _os, re as _re, time as _tm, gc as _gc

def _codecells(path):
    nb = _j.load(open(path))
    return [''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code']
def _cell(cells, m):
    h = [s for s in cells if s.lstrip().startswith(m)]
    assert len(h) == 1, f'marker not unique/found: {m!r} ({len(h)} hits)'
    return h[0]

_ccv, _cce = _codecells(NB_V12), _codecells(NB_E2)
_portable_candidates = [s for s in _codecells(NB_PORTABLE_SETUP)
                        if '# CTC local runtime' in s]
assert len(_portable_candidates) == 1, 'portable setup cell not found uniquely'
_portable_setup = _portable_candidates[0]
_legacy_setup_indices = [i for i, s in enumerate(_ccv)
                         if '# CTC local runtime' in s]
assert len(_legacy_setup_indices) == 1, 'sealed v1.2 setup cell not found uniquely'
_ccv[_legacy_setup_indices[0]] = _portable_setup

_h0pos = next(i for i, s in enumerate(_ccv) if s.lstrip().startswith('# ── H0.'))
SETUP = _ccv[:_h0pos]
SRC_H0, SRC_G2 = _cell(_ccv, '# ── H0.'), _cell(_ccv, '# ── G2.')
SRC_E21 = _cell(_cce, '# ── E2-1')
SRC_E22 = _cell(_cce, '# ── E2-2')
SRC_E23 = _cell(_cce, '# ── E2-3')
SRC_E24 = _cell(_cce, '# ── E2-4')
assert 'DP_S' in SRC_E22 and 'S_SEED' in SRC_E22, \
    'E2 notebook is not rev. C (no small-OOD arm) — update it before running AUTORUN'

_ip = get_ipython()
def _run(tag, src):
    t0 = _tm.time(); print(f'\n┏━━ E2 AUTORUN ▶ {tag} ' + '━' * 34)
    r = _ip.run_cell(src)
    if not r.success:
        print(f'┗━━ ✗ {tag} FAILED — aborting the whole run; report the error above.')
        r.raise_error()
    print(f'┗━━ ✓ {tag} done in {(_tm.time()-t0)/60:.1f} min')

def _regime_done(rg):
    d = f'{_KIT}/new-reward/checkpoints/conformal_controls_{rg}'
    man = f'{d}/e2_manifest_{rg}.json'
    if not (_os.path.exists(man) and _os.path.exists(f'{d}/e2_results_{rg}.csv')):
        return False
    try:
        m = _j.load(open(man))
        return ('s_arm' in m) and bool(m.get('gates', {}).get('e2b_training_determinism', False))
    except Exception:
        return False

_H0_RE = _re.compile(r"(?m)^PANEL_REGIME\s*=\s*'[^']+'")
print(f'plan: {RUN_REGIMES} | split={SPLIT_MODE} | per regime: H0, G2, E2-1..E2-3; then E2-4')
for _i, _s in enumerate(SETUP):
    _run(f'setup {_i+1}/{len(SETUP)}', _s)

# runtime capture — recorded by E2-3 into every regime manifest
import platform as _pf, subprocess as _sp, numpy as _np2, torch as _th2
try:
    _cpu_model = [l for l in _sp.run(['lscpu'], capture_output=True, text=True).stdout.splitlines()
                  if 'Model name' in l][0].split(':', 1)[1].strip()
except Exception:
    _cpu_model = 'unknown'
E2_RUNTIME = {'python': _pf.python_version(), 'numpy': _np2.__version__,
              'torch': _th2.__version__, 'threads': int(_th2.get_num_threads()),
              'cpu_model': _cpu_model}
print('runtime:', E2_RUNTIME)

_RESET = ['BATCH', 'TR_MASK', 'TE_MASK', 'EPS_E2TR', 'EPS_E2TE', 'pack_e2tr', 'pack_e2te',
          'SP_E2TE', 'DATES_E2TE', 'AUDIT', 'F_TE', 'F_TR', 'DP_OOD', 'DP_IND', 'DP_S',
          'E2_POLS', 'E2R', 'E2A_HASH_OK', 'EPS_S', 'pack_s']
_T0 = _tm.time()
for _rg in RUN_REGIMES:
    if SKIP_DONE and _regime_done(_rg):
        print(f'\n↷ {_rg}: rev. C e2 manifest + results already on disk — skipped (resume mode)')
        continue
    for _k in _RESET:
        _ip.user_ns.pop(_k, None)
    _gc.collect()
    _h0 = _H0_RE.sub(f"PANEL_REGIME = '{_rg}'", SRC_H0, count=1)
    assert f"PANEL_REGIME = '{_rg}'" in _h0, 'H0 regime patch failed'
    _run(f'{_rg} · H0', _h0)
    _run(f'{_rg} · G2', SRC_G2)
    _run(f'{_rg} · E2-1 split+audit', SRC_E21)
    _run(f'{_rg} · E2-2 gates + arms O/S', SRC_E22)
    _run(f'{_rg} · E2-3 three-arm compare+save', SRC_E23)
    print(f'\n★★ {_rg} complete — total elapsed {(_tm.time()-_T0)/60:.1f} min')
_run('E2-4 aggregation', SRC_E24)
print(f'\n━━ E2 AUTORUN COMPLETE in {(_tm.time()-_T0)/60:.1f} min ━━')
print('deliverable → new-reward/checkpoints/e2_summary.csv '
      '(+ per-regime e2_results / e2_audit / e2_manifest)')